# 05 — Correlation matrix design

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ai-systems-today/cubespec/blob/main/notebooks/05_correlation_presets.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ai-systems-today/cubespec/main?urlpath=lab/tree/notebooks/05_correlation_presets.ipynb) [![nbviewer](https://img.shields.io/badge/render-nbviewer-orange)](https://nbviewer.org/github/ai-systems-today/cubespec/blob/main/notebooks/05_correlation_presets.ipynb) [![Dashboard](https://img.shields.io/badge/open-dashboard-2563eb)](https://sensitive-spark.lovable.app)

**Abstract.** Apply the four built-in presets, verify positive-definiteness, and load a custom matrix from CSV.

**Mirrors.** **Parameters** tab → *Correlation* preset selector + *Load Corr CSV* control.


In [ ]:
# cubespec-bootstrap
# Install cubespec on first run (Colab/Binder safe; no-op if already importable).
try:
    import cubespec  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'git+https://github.com/ai-systems-today/cubespec.git'])
    import cubespec  # noqa: F401


## 1. Four correlation presets

The Parameters tab exposes four built-in presets:

* `identity`  — diagonal only; equivalent to independent sampling.
* `geometry`  — moderate +0.3 correlation between P1, P2, P3.
* `forces`    — strong +0.5 correlation between P4, P5, P6.
* `weak`      — global +0.2 between every off-diagonal pair.

Each preset must remain **positive-definite**, otherwise Cholesky fails and the dashboard shows a red warning. We verify that and compare the resulting P9 spread.


In [ ]:
import numpy as np, pandas as pd
from cubespec import DEFAULT_CSP, sample_mvn, compute_outputs_batch
from cubespec.correlation import (
    identity_corr, GEOMETRY_PRESET, FORCES_PRESET, is_positive_definite, cholesky,
)

WEAK = identity_corr(7) + 0.2 * (np.ones((7, 7)) - np.eye(7))
PRESETS = {
    'identity': identity_corr(7),
    'geometry': GEOMETRY_PRESET,
    'forces':   FORCES_PRESET,
    'weak':     WEAK,
}
rows = []
for name, R in PRESETS.items():
    pd_ok = is_positive_definite(R)
    Y = compute_outputs_batch(sample_mvn(DEFAULT_CSP, n=5_000, seed=1337, corr=R))[:, 2]
    rows.append({'preset': name, 'positive_definite': pd_ok,
                 'mean': Y.mean(), 'sd': Y.std(ddof=1)})
df = pd.DataFrame(rows).set_index('preset')
df


## 2. Visualise the four presets as heatmaps


In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, (name, R) in zip(axes, PRESETS.items()):
    im = ax.imshow(R, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_title(name)
    ax.set_xticks(range(7)); ax.set_yticks(range(7))
    ax.set_xticklabels([f'P{i}' for i in range(7)], fontsize=8)
    ax.set_yticklabels([f'P{i}' for i in range(7)], fontsize=8)
fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
plt.show()


## 3. Custom correlation from CSV (mirrors the Parameters tab Load Corr CSV control)


In [ ]:
from pathlib import Path
csv_path = Path('/tmp/corr_demo.csv')
labels = [f'P{i}' for i in range(7)]
with csv_path.open('w') as fh:
    fh.write(',' + ','.join(labels) + '\n')
    for r, lbl in enumerate(labels):
        fh.write(lbl + ',' + ','.join(f'{GEOMETRY_PRESET[r, c]:.2f}' for c in range(7)) + '\n')
loaded = pd.read_csv(csv_path, index_col=0).to_numpy()
print('loaded matches GEOMETRY_PRESET :', np.allclose(loaded, GEOMETRY_PRESET))
print('positive-definite              :', is_positive_definite(loaded))


In [ ]:
# cubespec-reproducibility-footer
import platform, sys, numpy as np, scipy, pandas as pd, cubespec
print('Reproducibility')
print('  cubespec :', cubespec.__version__)
print('  python   :', sys.version.split()[0], 'on', platform.system())
print('  numpy    :', np.__version__)
print('  scipy    :', scipy.__version__)
print('  pandas   :', pd.__version__)
